# 2. Перенос на MultiSports и MUVY

Здесь показаны уже выполненные внешние проверки и короткая адаптация моделей. Использовались 30 внешних test-клипов MultiSports, 30 trainval-клипов для адаптации и 15 клипов MUVY. Дополнительный ранний прогон на девяти клипах и smoke-тесты сохранены в `outputs/`, но не смешиваются с основными таблицами.

Код загрузки и подготовки: `sample_multisports.py`, `sample_muvy.py`, `prepare_multisports_adaptation.py`, `prepare_mixed_adaptation.py`. Обучение: `train.py`; внешняя оценка: `evaluate_external.py`.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "scripts/train_vk_bundle.py").is_file()
             and (p / "configs/experiment_index.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Запустите ноутбук внутри sports_video_repo")

def load_json(path):
    return json.loads((ROOT / path).read_text(encoding="utf-8"))

def load_jsonl(path):
    return [json.loads(line) for line in (ROOT / path).read_text(encoding="utf-8").splitlines() if line]

plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True})
print("Читаем сохранённые результаты; обучение и скачивание не запускаются.")

Читаем сохранённые результаты; обучение и скачивание не запускаются.


## Сохранённые внешние метрики

In [2]:
reports = ["external_multisports_30", "external_multisports_30_adapted", "external_multisports_30_mixed",
           "external_multisports_30_x3d_baseline", "external_multisports_30_x3d_mixed",
           "external_muvy_15_baseline", "external_muvy_15_adapted", "external_muvy_15_mixed",
           "external_muvy_15_x3d_baseline", "external_muvy_15_x3d_mixed"]
table = []
for run in reports:
    report = load_json(f"outputs/{run}/report.json")
    for model, result in report["models"].items():
        table.append({"проверка": run, "модель": model, "клипов": report["n_videos"],
                      "Macro-F1 наблюдаемых спортов": result["sport"]["macro_f1_observed_sports"],
                      "accuracy": result["sport"]["accuracy"],
                      "мс/клип с декодированием": result["runtime"]["ms_per_video"]})
display(pd.DataFrame(table).round(3))

,проверка,модель,клипов,Macro-F1 наблюдаемых спортов,accuracy,мс/клип с декодированием
0,external_multisports_30,dinov2_small,30,0.490,0.567,135.286
1,external_multisports_30,r3d_18,30,0.524,0.600,421.882
2,external_multisports_30_adapted,dinov2_small,30,0.663,0.667,139.923
3,external_multisports_30_adapted,r3d_18,30,0.898,0.900,422.380
4,external_multisports_30_mixed,dinov2_small,30,0.933,0.933,131.869
5,external_multisports_30_mixed,r3d_18,30,0.915,0.900,432.267
6,external_multisports_30_x3d_baseline,x3d_xs,30,0.540,0.633,149.343
7,external_multisports_30_x3d_mixed,x3d_xs,30,0.898,0.900,139.368
8,external_muvy_15_baseline,dinov2_small,15,0.296,0.533,230.479
9,external_muvy_15_baseline,r3d_18,15,0.413,0.467,813.928


## Как были устроены уже выполненные адаптации

In [3]:
runs = ["A1_dinov2_multisports_adapt", "A2_r3d18_multisports_adapt", "A3_dinov2_mixed",
        "A4_r3d18_mixed", "A5_x3d_xs_mixed"]
table = []
for run in runs:
    cfg = load_json(f"outputs/{run}/config.json")
    table.append({"запуск": run, "эпох": cfg["epochs"], "batch": cfg["batch_size"],
        "encoder заморожен": cfg["freeze"], "инициализация": cfg["init_from"],
        "train": cfg["train_manifest"], "validation": cfg["val_manifest"]})
display(pd.DataFrame(table))
display(pd.DataFrame([load_json("data/splits/adapt_summary.json"), load_json("data/splits/mixed_summary.json")]))

,запуск,эпох,batch,encoder заморожен,инициализация,train,validation
0,A1_dinov2_multisports_adapt,4,8,True,outputs/E3_dinov2_small_frozen_4f/checkpoint.pt,data/splits/adapt_train.csv,data/splits/adapt_val.csv
1,A2_r3d18_multisports_adapt,2,4,False,outputs/E5_r3d18_16f/checkpoint.pt,data/splits/adapt_train.csv,data/splits/adapt_val.csv
2,A3_dinov2_mixed,3,8,True,outputs/E3_dinov2_small_frozen_4f/checkpoint.pt,data/splits/mixed_train.csv,data/splits/mixed_val.csv
3,A4_r3d18_mixed,2,4,False,outputs/E5_r3d18_16f/checkpoint.pt,data/splits/mixed_train.csv,data/splits/mixed_val.csv
4,A5_x3d_xs_mixed,2,4,False,outputs/E4_x3d_xs_4f/checkpoint.pt,data/splits/mixed_train.csv,data/splits/mixed_val.csv


,seed,train_clips,val_clips,train_sports,val_sports,external_test_groups,group_intersection,multi_repeat,train_rows,val_rows,train_unique_clips,train_source_groups,val_source_groups,train_by_source,val_by_source
0,42,39.0,10.0,"{'tennis': 15, 'basketball': 8, 'football': 8,...","{'tennis': 4, 'basketball': 2, 'volleyball': 2...",30.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,42,NaN,NaN,NaN,NaN,NaN,NaN,3.0,369.0,66.0,321.0,84.0,19.0,"{'UCF101': 297, 'MultiSports': 72}","{'UCF101': 60, 'MultiSports': 6}"


In [4]:
baseline = load_json("outputs/external_multisports_30/report.json")
result = baseline["models"]["dinov2_small"]["sport"]
display(pd.DataFrame(result["confusion_matrix_all_four"], index=result["sport_order"], columns=result["sport_order"]))

,basketball,football,tennis,volleyball
basketball,0,0,0,10
football,0,10,0,0
tennis,0,0,0,0
volleyball,0,1,2,7


Внешние результаты показали, что успех на UCF101 не переносится автоматически на трансляции матчей. Баскетбол часто путался с волейболом. После адаптации улучшение зависит от источника; маленькие внешние выборки уже использовались для выбора следующего опыта, поэтому их нельзя считать нетронутым итоговым тестом. Две категории соревнования здесь ещё не обучались.

Подробности: [расширенный отчёт](../docs/results/STAGE_1_EXTENDED_RESULTS.md), параметры каждого обучения — в `outputs/A*/config.json`.